In [1]:
# Parameters
N_ESTIMATORS = 500


# 🏆 VFL Empire — GPU Training Notebook (papermill local)
Runs directly on VM via papermill — no Colab dependency.

In [2]:
# ── Parameters (can be overridden by papermill) ──
DATA_PATH  = '/home/ubuntu/faith-workspace/vfl-empire/data/vfl_training_data.csv'
MODELS_DIR = '/home/ubuntu/faith-workspace/vfl-empire/models'
N_ESTIMATORS = 500
ENSEMBLE_MIN_PROB = 0.70


In [3]:
import pandas as pd, numpy as np, pickle, json, os, shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb
import lightgbm as lgb
from datetime import datetime

df = pd.read_csv(DATA_PATH)
print(f'✅ Loaded {len(df):,} rows')


✅ Loaded 88,792 rows


In [4]:
le_pred   = LabelEncoder()
le_engine = LabelEncoder()
le_tier_h = LabelEncoder()
le_tier_a = LabelEncoder()

df['prediction_enc'] = le_pred.fit_transform(df['prediction'].fillna('Unknown'))
df['engine_enc']     = le_engine.fit_transform(df['engine'].fillna('Unknown'))
df['tier_home_enc']  = le_tier_h.fit_transform(df['tier_home'].fillna('mid'))
df['tier_away_enc']  = le_tier_a.fit_transform(df['tier_away'].fillna('mid'))

df['is_home_win'] = (df['prediction'] == 'Home Win').astype(int)
df['is_away_win'] = (df['prediction'] == 'Away Win').astype(int)
df['is_draw']     = df['prediction'].str.contains('Draw|DRAW|D$',regex=True).astype(int)
df['is_over']     = df['prediction'].str.contains('Over',case=False).astype(int)
df['is_under']    = df['prediction'].str.contains('Under',case=False).astype(int)
df['is_dnb']      = df['prediction'].str.contains('DNB',case=False).astype(int)
df['odds']            = pd.to_numeric(df['odds'],errors='coerce').fillna(1.5)
df['confidence']      = pd.to_numeric(df['confidence'],errors='coerce').fillna(50)
df['cv_1x2']          = pd.to_numeric(df['cv_1x2'],errors='coerce').fillna(0)
df['expected_value']  = (df['confidence']/100)*df['odds']-1
df['high_conf']       = (df['confidence']>=90).astype(int)
df['very_high_conf']  = (df['confidence']>=95).astype(int)

FEATURES = ['confidence','odds','cv_1x2','prediction_enc','engine_enc',
            'tier_home_enc','tier_away_enc','match_day',
            'is_home_win','is_away_win','is_draw','is_over','is_under','is_dnb',
            'expected_value','high_conf','very_high_conf']

X = df[FEATURES]; y = df['label']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')


Train: 71,033 | Test: 17,759


In [5]:
print(f'🚀 Training XGBoost ({N_ESTIMATORS} estimators)...')
xgb_model = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS, max_depth=6, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    tree_method='hist', random_state=42, verbosity=0,
    eval_metric='logloss', early_stopping_rounds=30
)
xgb_model.fit(X_train,y_train,eval_set=[(X_test,y_test)],verbose=False)
xgb_probs = xgb_model.predict_proba(X_test)[:,1]
xgb_acc   = accuracy_score(y_test, xgb_model.predict(X_test))
print(f'✅ XGBoost: {xgb_acc*100:.2f}%')


🚀 Training XGBoost (500 estimators)...


✅ XGBoost: 72.17%


In [6]:
print(f'⚡ Training LightGBM ({N_ESTIMATORS} estimators)...')
lgb_model = lgb.LGBMClassifier(
    n_estimators=N_ESTIMATORS, max_depth=7, learning_rate=0.03,
    num_leaves=127, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=20, random_state=42, verbose=-1
)
lgb_model.fit(X_train,y_train,eval_set=[(X_test,y_test)],
    callbacks=[lgb.early_stopping(30),lgb.log_evaluation(-1)])
lgb_probs = lgb_model.predict_proba(X_test)[:,1]
lgb_acc   = accuracy_score(y_test, lgb_model.predict(X_test))
print(f'✅ LightGBM: {lgb_acc*100:.2f}%')


⚡ Training LightGBM (500 estimators)...


Training until validation scores don't improve for 30 rounds


Did not meet early stopping. Best iteration is:
[500]	valid_0's binary_logloss: 0.529781


✅ LightGBM: 73.63%


In [7]:
ensemble_probs = (xgb_probs + lgb_probs) / 2
ensemble_acc   = accuracy_score(y_test, (ensemble_probs>=0.5).astype(int))
print(f'✅ Ensemble: {ensemble_acc*100:.2f}%')

X_test_df = X_test.copy()
X_test_df['prob'] = ensemble_probs
X_test_df['true'] = y_test.values

print('\n=== High-Confidence Filter (30k Strategy) ===')
for t in [0.60,0.65,0.70,0.75,0.80,0.85,0.90]:
    mask = X_test_df['prob'] >= t
    if mask.sum() > 10:
        acc = accuracy_score(X_test_df.loc[mask,'true'],(X_test_df.loc[mask,'prob']>=0.5).astype(int))
        print(f'  Prob>={t:.0%}: {acc*100:.1f}% acc | {mask.sum():,} bets ({100*mask.sum()/len(X_test_df):.1f}%)')


✅ Ensemble: 72.95%

=== High-Confidence Filter (30k Strategy) ===
  Prob>=60%: 77.2% acc | 9,849 bets (55.5%)
  Prob>=65%: 79.8% acc | 8,383 bets (47.2%)
  Prob>=70%: 82.8% acc | 6,564 bets (37.0%)
  Prob>=75%: 86.4% acc | 4,624 bets (26.0%)
  Prob>=80%: 90.5% acc | 2,676 bets (15.1%)
  Prob>=85%: 95.4% acc | 1,270 bets (7.2%)
  Prob>=90%: 97.1% acc | 625 bets (3.5%)


In [8]:
Path(MODELS_DIR).mkdir(exist_ok=True)
old = Path(MODELS_DIR)/'meta_classifier.txt'
if old.exists(): shutil.copy(old, str(old)+'.bak')

xgb_model.save_model(f'{MODELS_DIR}/xgb_meta_v2.json')
lgb_model.booster_.save_model(f'{MODELS_DIR}/lgb_meta_v2.txt')
with open(f'{MODELS_DIR}/encoders_v2.pkl','wb') as f:
    pickle.dump({'prediction':le_pred,'engine':le_engine,
                 'tier_home':le_tier_h,'tier_away':le_tier_a,
                 'features':FEATURES}, f)

summary = {
    'xgb_accuracy':      round(float(xgb_acc),4),
    'lgb_accuracy':      round(float(lgb_acc),4),
    'ensemble_accuracy': round(float(ensemble_acc),4),
    'n_estimators':      N_ESTIMATORS,
    'training_rows':     int(len(X_train)),
    'test_rows':         int(len(X_test)),
    'features':          FEATURES,
    'trained_at':        datetime.now().isoformat()
}
with open(f'{MODELS_DIR}/training_summary_v2.json','w') as f:
    json.dump(summary,f,indent=2)

print('\n🏆 Models saved:')
for p in sorted(Path(MODELS_DIR).glob('*v2*')):
    print(f'  {p.name}: {p.stat().st_size//1024} KB')
print(json.dumps(summary,indent=2))



🏆 Models saved:
  encoders_v2.pkl: 1 KB
  lgb_meta_v2.txt: 3948 KB
  training_summary_v2.json: 0 KB
  xgb_meta_v2.json: 2821 KB
{
  "xgb_accuracy": 0.7217,
  "lgb_accuracy": 0.7363,
  "ensemble_accuracy": 0.7295,
  "n_estimators": 500,
  "training_rows": 71033,
  "test_rows": 17759,
  "features": [
    "confidence",
    "odds",
    "cv_1x2",
    "prediction_enc",
    "engine_enc",
    "tier_home_enc",
    "tier_away_enc",
    "match_day",
    "is_home_win",
    "is_away_win",
    "is_draw",
    "is_over",
    "is_under",
    "is_dnb",
    "expected_value",
    "high_conf",
    "very_high_conf"
  ],
  "trained_at": "2026-05-29T17:15:33.974460"
}
